# 06 - Cosine Similarity and Recommendation Engine

Notebook ini menghitung cosine similarity dari TF-IDF matrix dan membuat fungsi rekomendasi Content-Based Filtering. Rekomendasi dibuat berdasarkan kemiripan profil teks antar tempat.

## Konsep Cosine Similarity

Cosine similarity mengukur kemiripan dua vektor berdasarkan arah vektornya. Pada sistem ini, setiap tempat sudah direpresentasikan sebagai vektor TF-IDF, kemudian dibandingkan untuk mencari tempat yang memiliki profil teks paling mirip.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path("../data/processed")
TFIDF_DATASET_FILE = DATA_DIR / "tfidf_dataset.csv"
TFIDF_MATRIX_FILE = DATA_DIR / "tfidf_matrix.pkl"
SIMILARITY_MATRIX_FILE = DATA_DIR / "cosine_similarity_matrix.pkl"

OUTPUT_COLUMNS = ["name", "category", "rating", "address", "subtypes", "type", "price_estimate", "similarity_score"]
VALID_TYPES = {"wisata", "kuliner", "hotel", "oleh_oleh"}

In [ ]:
# Load dataset dan TF-IDF matrix.
df = pd.read_csv(TFIDF_DATASET_FILE)

with open(TFIDF_MATRIX_FILE, "rb") as file:
    tfidf_matrix = pickle.load(file)

if tfidf_matrix.shape[0] != len(df):
    raise AssertionError("Jumlah baris dataset dan TF-IDF matrix tidak sama.")

print("TF-IDF dataset shape:", df.shape)
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Type counts:")
print(df["type"].value_counts())

In [ ]:
# Hitung cosine similarity matrix.
cosine_sim_matrix = cosine_similarity(tfidf_matrix)
cosine_sim_matrix = np.nan_to_num(cosine_sim_matrix, nan=0.0, posinf=0.0, neginf=0.0)

print("Similarity matrix shape:", cosine_sim_matrix.shape)
print("Nilai similarity minimum:", cosine_sim_matrix.min().round(4))
print("Nilai similarity maksimum:", cosine_sim_matrix.max().round(4))

In [ ]:
# Simpan cosine similarity matrix agar siap digunakan pada tahap berikutnya.
with open(SIMILARITY_MATRIX_FILE, "wb") as file:
    pickle.dump(cosine_sim_matrix, file)

print(f"Saved: {SIMILARITY_MATRIX_FILE}")

In [ ]:
def find_place_index(place_name, dataset):
    """Mencari index tempat berdasarkan nama secara sederhana."""
    keyword = str(place_name).lower().strip()
    matches = dataset[dataset["name"].str.lower().str.contains(keyword, na=False)]

    if matches.empty:
        raise ValueError(f"Tempat tidak ditemukan: {place_name}")

    return int(matches.index[0])


def recommend_similar_places(
    place_name,
    dataset,
    similarity_matrix,
    top_n=5,
    pool_size=20,
    filter_type=None,
    random_state=None
):
    """
    Memberikan rekomendasi tempat mirip dengan randomized recommendation pool.
    Alur: ambil kandidat teratas, lalu pilih acak dari pool agar hasil tidak repetitif.
    """
    if filter_type is not None and filter_type not in VALID_TYPES:
        raise ValueError(f"filter_type harus salah satu dari {sorted(VALID_TYPES)} atau None.")

    place_index = find_place_index(place_name, dataset)
    scores = similarity_matrix[place_index].copy()
    scores[place_index] = -1  # Hindari merekomendasikan tempat yang sama.

    candidates = dataset.copy()
    candidates["similarity_score"] = scores

    if filter_type is not None:
        candidates = candidates[candidates["type"] == filter_type]

    candidates = candidates[candidates["similarity_score"] > 0]
    candidates = candidates.sort_values("similarity_score", ascending=False)
    top_pool = candidates.head(pool_size)

    if len(top_pool) <= top_n:
        selected = top_pool
    else:
        selected = top_pool.sample(n=top_n, random_state=random_state)
        selected = selected.sort_values("similarity_score", ascending=False)

    return selected[OUTPUT_COLUMNS].reset_index(drop=True)

In [ ]:
# Contoh top similarity tanpa randomisasi untuk satu tempat.
example_place = df.iloc[0]["name"]
example_index = find_place_index(example_place, df)
example_scores = cosine_sim_matrix[example_index].copy()
example_scores[example_index] = -1

top_examples = df.copy()
top_examples["similarity_score"] = example_scores
top_examples = top_examples.sort_values("similarity_score", ascending=False).head(10)

print("Example place:", example_place)
top_examples[OUTPUT_COLUMNS]

In [ ]:
# Sample recommendation untuk masing-masing kategori.
def get_clean_example(dataset, category_type):
    """Memilih contoh input yang rapi untuk tampilan screenshot."""
    subset = dataset[
        (dataset["type"] == category_type)
        & (~dataset["name"].str.lower().str.contains("duplicate", na=False))
    ]
    return subset.iloc[0]["name"]


examples = {
    "wisata": get_clean_example(df, "wisata"),
    "kuliner": get_clean_example(df, "kuliner"),
    "hotel": get_clean_example(df, "hotel"),
    "oleh_oleh": "Krisna Oleh-Oleh Nusantara Jogja",
}

for category_type, place_name in examples.items():
    print(f"\nRekomendasi {category_type} untuk: {place_name}")
    display(recommend_similar_places(
        place_name,
        df,
        cosine_sim_matrix,
        top_n=5,
        pool_size=20,
        filter_type=category_type,
        random_state=42
    ))

## Keterbatasan Sistem

Rekomendasi hanya menggunakan Content-Based Filtering berdasarkan kemiripan teks. Harga pada dataset adalah estimasi statis, bukan harga real-time. Sistem belum menghitung transportasi secara langsung, belum menggunakan live API, dan belum melakukan optimasi rute terpendek. Karena menggunakan randomized recommendation pool, hasil rekomendasi dapat berbeda pada beberapa percobaan.